# Direct-order LGT Resume Pipeline

?? `RESUME_RUN_ID`? complete Trainer checkpoint?? ?? ??? ?, checkpoint ??? best ??/???? ?????.


In [ ]:
# 1) Install dependencies, then restart runtime once
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_lgt_multitask_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "jedi",
        "pandas==2.2.2",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")

In [ ]:
# 2) Setup + data unzip
from google.colab import drive
drive.mount("/content/drive")

import ast
import gc
import glob
import itertools
import json
import math
import os
import random
import re
import zipfile
from collections import defaultdict, deque
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, Subset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel, prepare_model_for_kbit_training

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
PAIRWISE_BEST_ADAPTER_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_pairwise_v1/runs/20260712_014247_A_basic_full_1epoch/best_exact_adapter"

# Resume the existing direct/LGT multitask run instead of creating a new RUN_ID.
RESUME_RUN_ID = "20260712_234828"
RESUME_CHECKPOINT_NAME = None  # None = use latest complete checkpoint, or set e.g. "checkpoint-300"
REQUIRE_RESUME_CHECKPOINT = True

LGT_ROOT_CANDIDATES = [
    "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_direct_order_multitask_v1",
    "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_multitask_v1",
]


def resolve_existing_output_dir(run_id):
    checked = []
    for root in LGT_ROOT_CANDIDATES:
        candidate = os.path.join(root, "runs", run_id, "lgt_multitask")
        checked.append(candidate)
        if os.path.isdir(candidate):
            return root, candidate
    raise RuntimeError(
        "Could not find the existing LGT run. Checked:\n" + "\n".join(checked)
    )


LGT_ROOT, OUTPUT_DIR = resolve_existing_output_dir(RESUME_RUN_ID)
RUN_ID = RESUME_RUN_ID
RUN_ROOT = os.path.dirname(OUTPUT_DIR.rstrip("/"))
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_direct_order_best.csv")
if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

os.makedirs(EVAL_DIR, exist_ok=True)

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(PAIRWISE_BEST_ADAPTER_DIR, "adapter_config.json")), PAIRWISE_BEST_ADAPTER_DIR

SEED = 42
VALID_RATIO = 0.1

# Full 1-epoch run. Set small integers only for smoke tests.
TRAIN_ROWS = None
VALID_ROWS = None

TASK_RATIOS = {
    "pairwise": 0.40,
    "first": 0.15,
    "last": 0.15,
    "order": 0.30,
}

TASK_LOSS_WEIGHTS = {
    "pairwise": 1.0,
    "first": 1.0,
    "last": 1.0,
    "order": 1.0,
}

# If True, the sampled training list contains every pairwise relation at least once.
PRESERVE_ALL_PAIRWISE = True

MAX_TRAIN_STEPS = None
SAVE_STEPS = 250
LEARNING_RATE = 2e-5
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
TASK_LOSS_LOG_STEPS = 20

PAIRWISE_EVAL_ROWS = 200
FIRST_LAST_EVAL_ROWS = 300
ORDER_EVAL_ROWS = 300
TEST_INFERENCE_ROWS = None

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
SMOKE_TEST = False

PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

def reset_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

reset_all_seeds(SEED)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("data:", DATA_DIR)
print("pairwise adapter:", PAIRWISE_BEST_ADAPTER_DIR)
print("resume run id:", RUN_ID)
print("resume output:", OUTPUT_DIR)
print("resume checkpoint name:", RESUME_CHECKPOINT_NAME)


In [ ]:
# 3) Data split + local-to-global multitask record generation
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result

def format_order(answer):
    return "[" + ", ".join(str(int(value)) for value in answer) + "]"

def parse_order_prediction(text):
    """Parse outputs such as [3, 4, 1, 2] or [Image 3, Image 4, Image 1, Image 2]."""
    normalized = str(text).strip()
    normalized = re.sub(r"(?i)\\b(?:input|image)\\s*", "", normalized)
    match = re.search(
        r"\\[\\s*([1-4])\\s*,\\s*([1-4])\\s*,\\s*([1-4])\\s*,\\s*([1-4])\\s*\\]",
        normalized,
    )
    if not match:
        return None
    values = [int(value) for value in match.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None

def parse_digit_prediction(text, max_digit):
    match = re.fullmatch(r"\s*([1-%d])\s*" % max_digit, str(text))
    if not match:
        return None
    return int(match.group(1))

def order_to_sequence(answer):
    # Answer is ranks per Input. Return input numbers in chronological order.
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]

def first_target(answer):
    return str(order_to_sequence(answer)[0])

def last_target(answer):
    return str(order_to_sequence(answer)[-1])

def pairwise_target(answer, first_index, second_index):
    return "1" if int(answer[first_index]) < int(answer[second_index]) else "2"

def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()

def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]

train_df = pd.read_csv(TRAIN_CSV)
train_df["Id"] = train_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

test_df = pd.read_csv(TEST_CSV)
test_df["Id"] = test_df["Id"].astype(str)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)

if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

assert set(training_df["Id"]).isdisjoint(set(validation_df["Id"]))

def build_task_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    for row_index, row in dataframe.reset_index(drop=True).iterrows():
        answer = [int(value) for value in row["Answer_list"]]
        for pair_index, (first_index, second_index) in enumerate(PAIR_INDICES):
            pools["pairwise"].append({
                "task_type": "pairwise",
                "row_index": row_index,
                "pair_index": pair_index,
                "first_index": first_index,
                "second_index": second_index,
                "target": pairwise_target(answer, first_index, second_index),
            })
        pools["first"].append({"task_type": "first", "row_index": row_index, "target": first_target(answer)})
        pools["last"].append({"task_type": "last", "row_index": row_index, "target": last_target(answer)})
        pools["order"].append({"task_type": "order", "row_index": row_index, "target": format_order(order_to_sequence(answer))})
    return pools

def balanced_records_from_pools(pools, ratios, seed=42, preserve_all_pairwise=True):
    rng = random.Random(seed)
    if preserve_all_pairwise:
        pair_count = len(pools["pairwise"])
        total = math.ceil(pair_count / ratios["pairwise"])
    else:
        total = len(pools["order"]) * 8

    target_counts = {task: max(1, int(round(total * ratio))) for task, ratio in ratios.items()}
    if preserve_all_pairwise:
        target_counts["pairwise"] = len(pools["pairwise"])

    records = []
    for task, target_count in target_counts.items():
        pool = list(pools[task])
        if not pool:
            continue
        if target_count <= len(pool):
            records.extend(rng.sample(pool, target_count))
        else:
            records.extend(pool)
            records.extend(rng.choice(pool) for _ in range(target_count - len(pool)))
    rng.shuffle(records)
    return records, target_counts

train_pools = build_task_pools(training_df)
train_records, train_target_counts = balanced_records_from_pools(
    train_pools,
    TASK_RATIOS,
    seed=SEED,
    preserve_all_pairwise=PRESERVE_ALL_PAIRWISE,
)

print("training rows:", len(training_df), "validation rows:", len(validation_df), "test rows:", len(test_df))
print("train record count:", len(train_records))
print("target task counts:", train_target_counts)
print(pd.Series([record["task_type"] for record in train_records]).value_counts(normalize=True).rename("ratio"))

In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(task_type, sentence):
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the beginning of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "last":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the end of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Arrange all images in chronological order.\n"
            "Return only one Python-style list of image numbers, such as [1, 2, 3, 4].\n"
            "Do not output any explanation."
        )
    raise ValueError(f"Unknown task_type: {task_type}")

def make_messages(example):
    content = []
    for label in example["image_labels"]:
        content.append({"type": "text", "text": f"\n{label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + example["instruction"]})
    return [{"role": "user", "content": content}]

class LGTMultiTaskDataset(Dataset):
    def __init__(self, dataframe, records, image_root, seed=42, augment_pairwise_flip=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.records = list(records)
        self.image_root = image_root
        self.seed = seed
        self.augment_pairwise_flip = augment_pairwise_flip

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = dict(self.records[index])
        row = self.dataframe.iloc[record["row_index"]]
        sample_id = str(row["Id"])
        sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
        all_image_paths = row_image_paths(row, self.image_root)
        task_type = record["task_type"]

        if task_type == "pairwise":
            first_index = int(record["first_index"])
            second_index = int(record["second_index"])
            target = str(record["target"])
            flip_rng = random.Random(self.seed * 100000 + index)
            if self.augment_pairwise_flip and flip_rng.random() < 0.5:
                first_index, second_index = second_index, first_index
                target = "1" if target == "2" else "2"
            image_paths = [all_image_paths[first_index], all_image_paths[second_index]]
            image_labels = ["First image", "Second image"]
        else:
            image_paths = all_image_paths
            image_labels = [f"Image {i}" for i in range(1, 5)]
            target = str(record["target"])

        return {
            "Id": sample_id,
            "task_type": task_type,
            "image_paths": image_paths,
            "image_labels": image_labels,
            "instruction": task_instruction(task_type, sentence),
            "target": target,
            "answer_list": [int(value) for value in row["Answer_list"]],
        }

class QwenLGTCollator:
    def __init__(self, processor):
        self.processor = processor
        self.assistant_prefix_ids = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def find_last_subsequence(self, sequence, pattern):
        for start in range(len(sequence) - len(pattern), -1, -1):
            if sequence[start:start + len(pattern)] == pattern:
                return start
        return -1

    def __call__(self, examples):
        if len(examples) != 1:
            raise ValueError("Use batch size 1 with this collator.")
        example = examples[0]
        images = [load_rgb(path) for path in example["image_paths"]]
        messages = make_messages(example) + [{"role": "assistant", "content": example["target"]}]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        model_inputs = self.processor(text=[text], images=images, padding=False, return_tensors="pt")

        input_ids = model_inputs["input_ids"][0].tolist()
        assistant_pos = self.find_last_subsequence(input_ids, self.assistant_prefix_ids)
        if assistant_pos >= 0:
            answer_start = assistant_pos + len(self.assistant_prefix_ids)
        else:
            prompt_text = self.processor.apply_chat_template(make_messages(example), tokenize=False, add_generation_prompt=True)
            prompt_inputs = self.processor(text=[prompt_text], images=images, padding=False, return_tensors="pt")
            answer_start = prompt_inputs["input_ids"].shape[1]

        labels = model_inputs["input_ids"].clone()
        labels[:, :answer_start] = -100
        if "attention_mask" in model_inputs:
            labels[model_inputs["attention_mask"] == 0] = -100
        model_inputs["labels"] = labels
        model_inputs["task_type"] = example["task_type"]
        return model_inputs

train_dataset = LGTMultiTaskDataset(training_df, train_records, TRAIN_IMAGE_DIR, seed=SEED)
print("example:", train_dataset[0]["task_type"], train_dataset[0]["target"])

In [ ]:
# 5) Model loading from Pairwise best adapter + task-loss Trainer
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def disable_sampling_warnings(model):
    generation_config = getattr(model, "generation_config", None)
    if generation_config is None:
        return
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None

def load_pairwise_adapter_for_multitask(trainable=True):
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    if trainable:
        base_model.config.use_cache = False
        base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base_model, PAIRWISE_BEST_ADAPTER_DIR, is_trainable=trainable)
    model.config.use_cache = not trainable
    disable_sampling_warnings(model)
    return model

class TaskLossTrainer(Trainer):
    def __init__(self, *args, task_loss_log_steps=20, task_loss_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.task_loss_log_steps = task_loss_log_steps
        self.task_loss_weights = task_loss_weights or {}
        self._task_loss_buffer = defaultdict(list)
        self._weighted_task_loss_buffer = defaultdict(list)
        self._last_task_loss_log_step = -1

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        task_type = inputs.pop("task_type", None)
        outputs = model(**inputs)
        loss = outputs.loss
        task_name = task_type if isinstance(task_type, str) else "unknown"
        task_weight = float(self.task_loss_weights.get(task_name, 1.0))
        weighted_loss = loss * task_weight
        self._task_loss_buffer[task_name].append(float(loss.detach().cpu()))
        self._weighted_task_loss_buffer[task_name].append(float(weighted_loss.detach().cpu()))
        step = int(getattr(self.state, "global_step", 0))
        if step > 0 and step % self.task_loss_log_steps == 0 and step != self._last_task_loss_log_step:
            logs = {}
            for name, values in list(self._task_loss_buffer.items()):
                if values:
                    logs[f"train_{name}_loss"] = float(np.mean(values))
            for name, values in list(self._weighted_task_loss_buffer.items()):
                if values:
                    logs[f"train_{name}_weighted_loss"] = float(np.mean(values))
            if logs:
                self.log(logs)
            self._task_loss_buffer.clear()
            self._weighted_task_loss_buffer.clear()
            self._last_task_loss_log_step = step
        return (weighted_loss, outputs) if return_outputs else weighted_loss

model = load_pairwise_adapter_for_multitask(trainable=True)
trainable_parameter_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
print("trainable parameter module count:", len(trainable_parameter_names))
model.print_trainable_parameters()


In [ ]:
# 6) Resume local-to-global multitask training from the existing run checkpoint
reset_all_seeds(SEED)


def checkpoint_step(path):
    base = os.path.basename(path.rstrip("/"))
    if base.startswith("checkpoint-"):
        return int(base.split("-")[-1])
    return -1


def is_complete_trainer_checkpoint(path):
    required_files = [
        "adapter_config.json",
        "trainer_state.json",
        "optimizer.pt",
        "scheduler.pt",
    ]
    return os.path.isdir(path) and all(os.path.exists(os.path.join(path, name)) for name in required_files)


def resolve_resume_checkpoint():
    if RESUME_CHECKPOINT_NAME:
        checkpoint = os.path.join(OUTPUT_DIR, RESUME_CHECKPOINT_NAME)
        if not is_complete_trainer_checkpoint(checkpoint):
            missing = [
                name for name in ["adapter_config.json", "trainer_state.json", "optimizer.pt", "scheduler.pt"]
                if not os.path.exists(os.path.join(checkpoint, name))
            ]
            raise RuntimeError(f"Incomplete resume checkpoint: {checkpoint}. Missing: {missing}")
        return checkpoint

    checkpoints = sorted(
        glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
        key=checkpoint_step,
        reverse=True,
    )
    complete = [path for path in checkpoints if is_complete_trainer_checkpoint(path)]
    if complete:
        return complete[0]
    if REQUIRE_RESUME_CHECKPOINT:
        checked = "\n".join(checkpoints[:10]) if checkpoints else "(no checkpoint-* folders found)"
        raise RuntimeError("No complete Trainer checkpoint found under OUTPUT_DIR. Checked:\n" + checked)
    return None


resume_checkpoint = resolve_resume_checkpoint()
print("Resuming from:", resume_checkpoint)

if SMOKE_TEST:
    train_data = Subset(train_dataset, range(min(40, len(train_dataset))))
    max_steps = 20
    save_steps = 20
else:
    train_data = train_dataset
    max_steps = -1 if MAX_TRAIN_STEPS is None else MAX_TRAIN_STEPS
    save_steps = SAVE_STEPS

training_argument_values = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 1,
    "max_steps": max_steps,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": 60 if max_steps == -1 else max(1, int(max_steps * 0.05)),
    "max_grad_norm": 0.3,
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "logging_steps": 20,
    "save_strategy": "steps",
    "save_steps": save_steps,
    "save_total_limit": None,
    "report_to": "none",
    "remove_unused_columns": False,
    "dataloader_num_workers": 0,
    "seed": SEED,
    "data_seed": SEED,
    "label_names": ["labels"],
}
if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
    training_argument_values["eval_strategy"] = "no"
else:
    training_argument_values["evaluation_strategy"] = "no"

trainer = TaskLossTrainer(
    model=model,
    args=TrainingArguments(**training_argument_values),
    train_dataset=train_data,
    data_collator=QwenLGTCollator(processor),
    task_loss_log_steps=TASK_LOSS_LOG_STEPS,
    task_loss_weights=TASK_LOSS_WEIGHTS,
)
trainer.train(resume_from_checkpoint=resume_checkpoint)
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

resume_info = {
    "run_id": RUN_ID,
    "output_dir": OUTPUT_DIR,
    "resumed_from_checkpoint": resume_checkpoint,
    "pairwise_best_adapter_dir": PAIRWISE_BEST_ADAPTER_DIR,
    "train_rows": len(training_df),
    "validation_rows": len(validation_df),
    "task_ratios": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "preserve_all_pairwise": PRESERVE_ALL_PAIRWISE,
    "train_target_counts": train_target_counts,
    "max_train_steps": MAX_TRAIN_STEPS,
    "save_steps": SAVE_STEPS,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
}
with open(os.path.join(OUTPUT_DIR, "resume_run_config.json"), "w", encoding="utf-8") as f:
    json.dump(resume_info, f, ensure_ascii=False, indent=2)

del model
trainer = None
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 7) Validation helpers for pairwise / first / last / full-order
def load_adapter_for_eval(adapter_dir):
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    loaded_model = PeftModel.from_pretrained(base_model, adapter_dir)
    loaded_model.eval()
    loaded_model.config.use_cache = True
    disable_sampling_warnings(loaded_model)
    return loaded_model

def make_eval_example(row, task_type, pair=None, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    all_paths = row_image_paths(row, image_root)
    answer = [int(value) for value in row["Answer_list"]] if "Answer_list" in row else None
    if task_type == "pairwise":
        first_index, second_index = pair
        image_paths = [all_paths[first_index], all_paths[second_index]]
        image_labels = ["First image", "Second image"]
        target = pairwise_target(answer, first_index, second_index)
    else:
        image_paths = all_paths
        image_labels = [f"Image {i}" for i in range(1, 5)]
        if task_type == "first":
            target = first_target(answer)
        elif task_type == "last":
            target = last_target(answer)
        elif task_type == "order":
            target = format_order(order_to_sequence(answer))
        else:
            target = None
    return {
        "Id": sample_id,
        "task_type": task_type,
        "image_paths": image_paths,
        "image_labels": image_labels,
        "instruction": task_instruction(task_type, sentence),
        "target": target,
        "answer_list": answer,
    }

@torch.no_grad()
def generate_text(model, example, max_new_tokens=16):
    text = processor.apply_chat_template(make_messages(example), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        inputs = processor(text=[text], images=images, return_tensors="pt").to(model.device)
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        output_ids = generated_ids[0, inputs.input_ids.shape[1]:]
        return processor.decode(output_ids, skip_special_tokens=True).strip()
    finally:
        processor.tokenizer.padding_side = old_padding_side

def relation_pair_accuracy(pred_order_inputs, answer_ranks):
    if pred_order_inputs is None:
        return 0, len(PAIR_INDICES)
    predicted_ranks = [0] * 4
    for position, input_number in enumerate(pred_order_inputs, start=1):
        predicted_ranks[input_number - 1] = position
    correct = 0
    for first_index, second_index in PAIR_INDICES:
        pred_first_earlier = predicted_ranks[first_index] < predicted_ranks[second_index]
        gold_first_earlier = answer_ranks[first_index] < answer_ranks[second_index]
        correct += int(pred_first_earlier == gold_first_earlier)
    return correct, len(PAIR_INDICES)

@torch.no_grad()
def evaluate_pairwise(model, dataframe, limit_rows=PAIRWISE_EVAL_ROWS):
    rows = []
    eval_df = dataframe.iloc[:limit_rows].copy() if limit_rows is not None else dataframe.copy()
    for _, row in tqdm(list(eval_df.iterrows()), total=len(eval_df), desc="pairwise eval"):
        for pair in PAIR_INDICES:
            example = make_eval_example(row, "pairwise", pair=pair)
            output_text = generate_text(model, example, max_new_tokens=4)
            prediction = parse_digit_prediction(output_text, 2)
            rows.append({
                "Id": example["Id"],
                "pair": str(pair),
                "target": int(example["target"]),
                "output_text": output_text,
                "prediction": prediction,
                "correct": prediction == int(example["target"]),
            })
    df = pd.DataFrame(rows)
    return df, {"pairwise_accuracy": float(df["correct"].mean()) if not df.empty else np.nan}

@torch.no_grad()
def evaluate_first_last(model, dataframe, task_type, limit_rows=FIRST_LAST_EVAL_ROWS):
    rows = []
    eval_df = dataframe.iloc[:limit_rows].copy() if limit_rows is not None else dataframe.copy()
    for _, row in tqdm(list(eval_df.iterrows()), total=len(eval_df), desc=f"{task_type} eval"):
        example = make_eval_example(row, task_type)
        output_text = generate_text(model, example, max_new_tokens=4)
        prediction = parse_digit_prediction(output_text, 4)
        rows.append({
            "Id": example["Id"],
            "task_type": task_type,
            "target": int(example["target"]),
            "output_text": output_text,
            "prediction": prediction,
            "correct": prediction == int(example["target"]),
        })
    df = pd.DataFrame(rows)
    return df, {f"{task_type}_accuracy": float(df["correct"].mean()) if not df.empty else np.nan}

@torch.no_grad()
def evaluate_order(model, dataframe, limit_rows=ORDER_EVAL_ROWS):
    rows = []
    eval_df = dataframe.iloc[:limit_rows].copy() if limit_rows is not None else dataframe.copy()
    for _, row in tqdm(list(eval_df.iterrows()), total=len(eval_df), desc="order eval"):
        example = make_eval_example(row, "order")
        output_text = generate_text(model, example, max_new_tokens=24)
        prediction = parse_order_prediction(output_text)
        answer_order = order_to_sequence(example["answer_list"])
        valid = prediction is not None
        position_correct = sum(p == a for p, a in zip(prediction, answer_order)) if valid else 0
        rel_correct, rel_total = relation_pair_accuracy(prediction, example["answer_list"])
        first_correct = valid and prediction[0] == answer_order[0]
        last_correct = valid and prediction[-1] == answer_order[-1]
        both_boundaries_correct = bool(first_correct and last_correct)
        all_pair_relations_correct = bool(valid and rel_correct == rel_total)
        boundary_error = bool(valid and not both_boundaries_correct)
        relation_error = bool(valid and rel_correct < rel_total)
        assembly_error = bool(valid and both_boundaries_correct and rel_correct == rel_total and prediction != answer_order)
        rows.append({
            "Id": example["Id"],
            "output_text": output_text,
            "Prediction": str(prediction) if prediction else "",
            "Answer": str(answer_order),
            "valid_output": valid,
            "position_correct": position_correct,
            "position_total": 4,
            "exact_correct": int(valid and prediction == answer_order),
            "pair_correct": rel_correct,
            "pair_total": rel_total,
            "first_correct": int(first_correct),
            "last_correct": int(last_correct),
            "first_and_last_both_correct": int(both_boundaries_correct),
            "all_pair_relations_correct": int(all_pair_relations_correct),
            "boundary_error": int(boundary_error),
            "relation_error": int(relation_error),
            "assembly_error": int(assembly_error),
        })
    df = pd.DataFrame(rows)
    both = df[df["first_and_last_both_correct"] == 1]
    all_pairs = df[df["all_pair_relations_correct"] == 1]
    summary = {
        "valid_output_rate": float(df["valid_output"].mean()) if not df.empty else np.nan,
        "position_accuracy": float(df["position_correct"].sum() / df["position_total"].sum()) if not df.empty else np.nan,
        "order_pair_accuracy": float(df["pair_correct"].sum() / df["pair_total"].sum()) if not df.empty else np.nan,
        "exact_match_accuracy": float(df["exact_correct"].mean()) if not df.empty else np.nan,
        "first_accuracy_from_order": float(df["first_correct"].mean()) if not df.empty else np.nan,
        "last_accuracy_from_order": float(df["last_correct"].mean()) if not df.empty else np.nan,
        "first_and_last_both_correct": float(df["first_and_last_both_correct"].mean()) if not df.empty else np.nan,
        "all_pair_relations_correct": float(df["all_pair_relations_correct"].mean()) if not df.empty else np.nan,
        "boundary_error_rate": float(df["boundary_error"].mean()) if not df.empty else np.nan,
        "relation_error_rate": float(df["relation_error"].mean()) if not df.empty else np.nan,
        "assembly_error_rate": float(df["assembly_error"].mean()) if not df.empty else np.nan,
        "exact_match_given_correct_boundaries": float(both["exact_correct"].mean()) if not both.empty else np.nan,
        "exact_match_given_all_pair_relations": float(all_pairs["exact_correct"].mean()) if not all_pairs.empty else np.nan,
        "order_total": int(len(df)),
    }
    return df, summary


In [ ]:
# 8) Evaluate every saved checkpoint and choose the best by full-order exact match
def checkpoint_step(path):
    base = os.path.basename(path.rstrip("/"))
    if base.startswith("checkpoint-"):
        return int(base.split("-")[-1])
    return 10**18

def find_adapter_dirs(root_dir):
    checkpoints = sorted(glob.glob(os.path.join(root_dir, "checkpoint-*")), key=checkpoint_step)
    candidates = [path for path in checkpoints if os.path.exists(os.path.join(path, "adapter_config.json"))]
    if os.path.exists(os.path.join(root_dir, "adapter_config.json")):
        candidates.append(root_dir)
    return candidates

def evaluate_adapter(adapter_dir, prefix):
    model = load_adapter_for_eval(adapter_dir)
    try:
        pair_df, pair_summary = evaluate_pairwise(model, validation_df, PAIRWISE_EVAL_ROWS)
        first_df, first_summary = evaluate_first_last(model, validation_df, "first", FIRST_LAST_EVAL_ROWS)
        last_df, last_summary = evaluate_first_last(model, validation_df, "last", FIRST_LAST_EVAL_ROWS)
        order_df, order_summary = evaluate_order(model, validation_df, ORDER_EVAL_ROWS)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    pair_df.to_csv(os.path.join(EVAL_DIR, f"{prefix}_pairwise_predictions.csv"), index=False)
    first_df.to_csv(os.path.join(EVAL_DIR, f"{prefix}_first_predictions.csv"), index=False)
    last_df.to_csv(os.path.join(EVAL_DIR, f"{prefix}_last_predictions.csv"), index=False)
    order_df.to_csv(os.path.join(EVAL_DIR, f"{prefix}_order_predictions.csv"), index=False)
    summary = {}
    summary.update(pair_summary)
    summary.update(first_summary)
    summary.update(last_summary)
    summary.update(order_summary)
    summary["adapter_dir"] = adapter_dir
    return summary

print("Evaluating original pairwise adapter baseline...")
baseline_summary = evaluate_adapter(PAIRWISE_BEST_ADAPTER_DIR, "baseline_pairwise_adapter")
baseline_pairwise_accuracy = baseline_summary["pairwise_accuracy"]
print("baseline:", baseline_summary)

checkpoint_rows = []
for adapter_dir in find_adapter_dirs(OUTPUT_DIR):
    prefix = os.path.basename(adapter_dir.rstrip("/")) or "final"
    if prefix == os.path.basename(OUTPUT_DIR.rstrip("/")):
        prefix = "final"
    print("evaluating:", adapter_dir)
    summary = evaluate_adapter(adapter_dir, prefix)
    checkpoint_rows.append(summary)
    print(summary)

checkpoint_df = pd.DataFrame(checkpoint_rows)
checkpoint_df["pairwise_drop"] = checkpoint_df["pairwise_accuracy"] - baseline_pairwise_accuracy

if not checkpoint_df.empty:
    # Primary criterion: direct full-order exact match.
    # Ties are resolved by order pair accuracy, position accuracy, valid-output rate,
    # first/last accuracy, and finally retained pairwise accuracy.
    best_row = checkpoint_df.sort_values(
        [
            "exact_match_accuracy",
            "order_pair_accuracy",
            "position_accuracy",
            "valid_output_rate",
            "first_accuracy_from_order",
            "last_accuracy_from_order",
            "pairwise_accuracy",
        ],
        ascending=False,
    ).iloc[0]
    BEST_ADAPTER_DIR = best_row["adapter_dir"]
else:
    raise RuntimeError("No trained adapter checkpoints found to evaluate.")

baseline_df = pd.DataFrame([baseline_summary])
baseline_df.to_csv(os.path.join(EVAL_DIR, "baseline_summary.csv"), index=False)
checkpoint_df.to_csv(os.path.join(EVAL_DIR, "checkpoint_summary.csv"), index=False)

BEST_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "best_lgt_adapter")
print("BEST_ADAPTER_DIR:", BEST_ADAPTER_DIR)
print("BEST_OUTPUT_DIR:", BEST_OUTPUT_DIR)
display(checkpoint_df)

In [ ]:
# 9) Save selected best adapter
best_model = load_adapter_for_eval(BEST_ADAPTER_DIR)
best_model.save_pretrained(BEST_OUTPUT_DIR)
processor.save_pretrained(BEST_OUTPUT_DIR)
with open(os.path.join(BEST_OUTPUT_DIR, "selection_info.json"), "w", encoding="utf-8") as f:
    json.dump({
        "best_adapter_dir": BEST_ADAPTER_DIR,
        "best_output_dir": BEST_OUTPUT_DIR,
        "baseline_pairwise_accuracy": baseline_pairwise_accuracy,
        "selection_rule": "highest direct full-order exact_match_accuracy; tie-break by order pair/position/valid/first/last/pairwise",
    }, f, ensure_ascii=False, indent=2)
del best_model
gc.collect()
torch.cuda.empty_cache()
print("Best adapter saved:", BEST_OUTPUT_DIR)

In [ ]:
# 10) Single-call full-order test inference + submission
class LGTTestDataset(Dataset):
    def __init__(self, dataframe, image_root):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_root = image_root

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        sample_id = str(row["Id"])
        sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
        image_paths = row_image_paths(row, self.image_root)
        return {
            "Id": sample_id,
            "task_type": "order",
            "image_paths": image_paths,
            "image_labels": [f"Image {i}" for i in range(1, 5)],
            "instruction": task_instruction("order", sentence),
        }

@torch.no_grad()
def run_test_inference(adapter_dir, dataframe, limit_rows=TEST_INFERENCE_ROWS):
    model = load_adapter_for_eval(adapter_dir)
    dataset_df = dataframe.iloc[:limit_rows].copy() if limit_rows is not None else dataframe.copy()
    dataset = LGTTestDataset(dataset_df, TEST_IMAGE_DIR)
    rows = []
    try:
        for index in tqdm(range(len(dataset)), desc="test full-order inference"):
            example = dataset[index]
            output_text = generate_text(model, example, max_new_tokens=24)
            prediction = parse_order_prediction(output_text)
            if prediction is None:
                prediction = [1, 2, 3, 4]
            rows.append({
                "Id": example["Id"],
                "Answer": str(prediction),
                "raw_output": output_text,
            })
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    submission_df = pd.DataFrame(rows)[["Id", "Answer"]]
    debug_df = pd.DataFrame(rows)
    submission_df.to_csv(SUBMIT_PATH, index=False)
    debug_df.to_csv(os.path.join(OUTPUT_DIR, "submission_direct_order_best_debug.csv"), index=False)
    return submission_df, debug_df

submission_df, debug_df = run_test_inference(BEST_OUTPUT_DIR, test_df)
print("saved submission:", SUBMIT_PATH)
display(submission_df.head())

In [ ]:
# 11) Final artifact summary
print("Training output directory:", OUTPUT_DIR)
print("Checkpoint evaluation summary:", os.path.join(EVAL_DIR, "checkpoint_summary.csv"))
print("Saved best adapter:", BEST_OUTPUT_DIR)
print("Submission CSV:", SUBMIT_PATH)
print("Debug CSV:", os.path.join(OUTPUT_DIR, "submission_direct_order_best_debug.csv"))
